# Sparse Matrix Support and Graph Neural Networks in Needle

A citation graph connects a small fraction of all possible pairs of papers. In a dense adjacency matrix, each missing citation still occupies a floating-point slot. We extended Needle with a compressed sparse row (CSR) matrix and a differentiable operation that multiplies this matrix by dense features. The implementation runs in Needle's native CPU backend. We evaluated it through a two-layer graph convolutional network (GCN) on Cora, using the same model with dense adjacency as a reference.

On our arm64 macOS machine, CSR required 116,948 bytes for the normalized adjacency. The dense matrix required 29,333,056 bytes. Across three paired seeds, the median training epoch took about 32 ms with CSR and 121 ms with dense storage. Both versions obtained the same held-out classification accuracy for each seed. These measurements cover full training steps, including the weight projections, loss, backward pass and optimizer update. The notebook includes the checks and raw per-run results behind the comparison.

**Reproduction.** The private [source archive](needle_sparse_gcn_source.zip) is also embedded in the next cell. It contains the Needle implementation, Cora data, tests, and recorded measurements. Open this notebook in a Colab CPU runtime or a local Jupyter session and run the cells in order. The notebook extracts the source when needed, builds the native extension, runs the tests, and exercises a short paired training run. To save a separate source archive, enable the download option in the next cell.

In [1]:
#@title Private project source (embedded ZIP)
from pathlib import Path
import base64
import tempfile

SOURCE_B64 = '...'
source_bytes = base64.b64decode(SOURCE_B64, validate=True)
archive_path = Path.cwd() / "needle_sparse_gcn_source.zip"
if archive_path.exists() and archive_path.read_bytes() != source_bytes:
    # Keep a different existing archive intact.
    archive_dir = tempfile.TemporaryDirectory(prefix="needle-archive-")
    archive_path = Path(archive_dir.name) / archive_path.name
if not archive_path.exists():
    archive_path.write_bytes(source_bytes)
del source_bytes

DOWNLOAD_PRIVATE_SOURCE = False  #@param {type:"boolean"}
if DOWNLOAD_PRIVATE_SOURCE:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink, display
        display(FileLink(str(archive_path)))
    else:
        files.download(str(archive_path))
print(f"Private source ready ({archive_path.stat().st_size:,} bytes).")

Private source ready (203,584 bytes).


In [2]:
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

here = Path.cwd()
def is_project(path):
    return (path / "python/needle/sparse.py").is_file() and (path / "apps/gcn.py").is_file()

root = next((p for p in (here, here / "needle_sparse_gcn", here / "final_project")
             if is_project(p)), None)
if root is None:
    archive = archive_path
    if not archive.exists():
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f"Place the private source archive at {archive} and rerun.")
        print("Upload needle_sparse_gcn_source.zip")
        uploaded = files.upload()
        name = next((name for name in uploaded if name.endswith(".zip")), None)
        if name is None:
            raise FileNotFoundError("Please select the project ZIP.")
        archive = here / name
    extraction = tempfile.TemporaryDirectory(prefix="needle-source-")
    destination = Path(extraction.name).resolve()
    with zipfile.ZipFile(archive) as z:
        for entry in z.namelist():
            if not (destination / entry).resolve().is_relative_to(destination):
                raise ValueError("Unexpected path inside the archive")
        z.extractall(destination)
    root = destination / "needle_sparse_gcn"
    if not is_project(root):
        raise ValueError("The archive is missing the Needle project directory")

python = sys.executable
modules = ("numpy", "scipy", "pybind11", "pytest")
if any(importlib.util.find_spec(m) is None for m in modules) or not shutil.which("cmake"):
    subprocess.run([python, "-m", "pip", "install", "-q", "-r", str(root / "requirements.txt")],
                   check=True)
env = os.environ.copy()
env["PATH"] = str(Path(python).parent) + os.pathsep + env.get("PATH", "")
env["PYTHONPATH"] = os.pathsep.join([str(root / "python"), str(root)])
env["NEEDLE_BACKEND"] = "nd"
def run(*args):
    subprocess.run(args, cwd=root, env=env, check=True)

run("cmake", "-S", ".", "-B", "build", "-DCMAKE_BUILD_TYPE=Release", f"-DPython_EXECUTABLE={python}")
run("cmake", "--build", "build", "-j", "2")
print("Native CPU extension ready.")

-- The CXX compiler identification is AppleClang 21.0.0.21000101
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done


-- Found Python: /Users/chiyixiao/study/10714/final_project/.venv/bin/python (found version "3.12.14") found components: Interpreter Development.Module
-- Performing Test HAS_FLTO_THIN
-- Performing Test HAS_FLTO_THIN - Success
-- Performing Test HAS_FLTO


-- Performing Test HAS_FLTO - Success
-- Found pybind11: /Users/chiyixiao/study/10714/final_project/.venv/lib/python3.12/site-packages/pybind11/include (found version "3.1.0")
-- Configuring done (0.8s)
-- Generating done (0.0s)
-- Build files have been written to: /private/var/folders/n0/w6gm76jn2qx81f6vm2lhhdl80000gn/T/needle-source-5m8devbb/needle_sparse_gcn/build
[ 50%] Building CXX object CMakeFiles/ndarray_backend_cpu.dir/src/ndarray_backend_cpu.cc.o


[100%] Linking CXX shared module /private/var/folders/n0/w6gm76jn2qx81f6vm2lhhdl80000gn/T/needle-source-5m8devbb/needle_sparse_gcn/python/needle/backend_ndarray/ndarray_backend_cpu.cpython-312-darwin.so


[100%] Built target ndarray_backend_cpu
Native CPU extension ready.


## Storage and interface

A CSR matrix of shape $m \times n$ contains three buffers. For row $i$, the integer offsets at positions $i$ and $i+1$ delimit a slice of the column-index and value arrays. An empty row has identical adjacent offsets. The final offset gives the number of stored entries, $s$. Our CPU backend represents each value as a 32-bit float and each index or offset as a 32-bit integer. Persistent payload size is therefore $8s+4(m+1)$ bytes. This count excludes object headers, temporaries and allocator overhead. We measure process memory separately.

The direct constructor takes a Needle tensor of values and preserves its connection to the computation graph. Rows and columns identify the positions to be updated and remain fixed. The coordinate-list constructor accepts host data, sorts entries into row order, sums weights at repeated coordinates and discards exact zero sums. Its result has a new leaf tensor of values in canonical CSR order. A caller who wants the weight at a particular edge to be trainable can inspect its gradient in that tensor. An update to a stored value does not require building a new pattern. A change in the set of coordinates requires a new pattern.

For $Y=AX$ with $X$ of shape $n \times f$, the forward kernel visits every stored $(i,j)$, multiplies the feature vector in row $j$ by its weight, and accumulates that contribution into output row $i$. The output is dense because even a single neighbor may contribute to every feature of a node. Initializing the output costs $O(mf)$, and accumulating the entries costs $O(sf)$. The kernel accesses contiguous feature rows through Needle's CPU buffers. A strided input view is compacted before passing its pointer to C++.

## Reverse-mode differentiation

Let $G$ be the upstream gradient with the same shape as $Y$. Perturbing input row $X_j$ changes each output row connected to column $j$. Its gradient is the weighted sum of those rows of $G$, giving $dX=A^T G$. The reverse kernel traverses the original CSR rows and accumulates into the destination indexed by each stored column. It can use exactly the same row offsets and column indices that the forward pass read.

Perturbing a single stored scalar $v_e=A_{ij}$ has a different effect. It changes only output row $i$, in the direction of the entire input vector $X_j$. The scalar gradient is therefore $dv_e=\langle G_i,X_j\rangle$. Computing one dot product per stored position gives a length-$s$ result without allocating a dense matrix of possible edge gradients. The two operations appear in the graph as Needle tensor operations, and the existing autograd engine handles accumulation when values or features have more than one consumer.

A hand calculation exercises the transpose and the value derivative at once. Use a two-by-three matrix with rows $(0,2,0)$ and $(3,0,4)$, and a column of features $(1,2,-1)^T$. The output is $(4,-1)^T$. Weighting the second output twice as strongly as the first gives an input gradient $(6,2,8)^T$ and stored-value gradients $(2,2,-2)$. An implementation that multiplies by the original matrix during feature backpropagation cannot pass this rectangular example.

## Data and learning objective

Cora supplies a bag-of-words feature vector and a topic label for each paper. We read the original Planetoid-format files distributed with the GCN reference implementation, verify their digests in the experiment driver, and use SciPy to deserialize the feature matrices. The test block uses its own order of node IDs. The loader scatters its features and labels to the IDs in the test-index file before constructing masks. A test examines rows of the original files to catch the case where features are sorted independently from their labels.

There are 2,708 nodes, 1,433 input features and seven classes. The public split supervises training on 140 labeled nodes, reserves 500 for validation and evaluates on 1,000 test nodes. Other nodes can still contribute features through message passing. This is transductive classification: the graph is available throughout training, while the loss uses labels from the training split. We compute a cross-entropy term per node, weight each term by the binary training mask and divide the sum by the number of training nodes. Changes to held-out labels must leave the training loss and its gradient unchanged.

We convert citations to undirected binary edges and deduplicate them before adding one self-loop per node. With $D$ the diagonal matrix of degrees after adding loops, each retained edge receives weight $1/\sqrt{D_{ii}D_{jj}}$. Both storage formats are built from the same list of normalized entries. The feature rows are divided by their sums; empty feature rows remain zero. These conventions give both implementations the same model input down to individual edge weights.

Following Kipf and Welling, the network applies two graph convolutions. A layer first projects features using $XW$, then aggregates the projected vectors with the normalized adjacency. The first projection reduces 1,433 features to 16, so this order substantially reduces the number of scalar operations in aggregation. A ReLU and training-time dropout separate the two layers. The second layer produces seven class scores per node. The experiment uses bias-free weights, Xavier initialization, Adam with learning rate 0.01 and weight decay 0.0005, and hidden dropout probability 0.5.

For this task the normalized graph is fixed. The inherited backward loop visits every tensor input when an operation participates in differentiation. Supplying a constant adjacency as an ordinary dense input could cause the engine to materialize an unused gradient the size of the graph. Both comparison paths therefore capture the adjacency as a fixed operand and expose the feature matrix as the only differentiable input to aggregation. The first feature projection uses the same fixed-input treatment in both models. The general CSR operator with differentiable edge weights is tested independently.

## Correctness checks

The operator tests compare the sparse implementation with dense Needle on empty rows, duplicates and rectangular matrices with several feature columns. A nonuniform upstream gradient makes wrong row and column indices visible. The finite-difference test perturbs each stored value and each feature in a small nonlinear scalar objective. Different error tolerances apply to this numerical approximation and to comparisons with analytical gradients. Another test passes transposed arrays to the fixed operators to exercise the distinction between logical tensor shape and physical strides.

At the model level, a small graph with fixed weights permits an independent NumPy calculation of logits, masked cross entropy, both parameter gradients and the result of an SGD update. Both Needle paths must match those quantities. A separate Cora comparison uses identical initial weights with dropout disabled and records the largest absolute difference for each quantity in a real forward/backward step. On the recorded Cora run, both losses were 1.94567168 and the largest measured difference was zero for logits, loss, both weight gradients and both updated weight matrices. These checks can detect errors that predicted classes would hide: a layer can move a logit substantially without changing the class with the largest score.

In [3]:
run(python, "-m", "pytest", "-q", "tests")
scratch = tempfile.TemporaryDirectory(prefix="needle-results-")
scratch_dir = Path(scratch.name)
run(python, "-m", "apps.verify_cora", "--output", str(scratch_dir / "correctness.json"))
print((scratch_dir / "correctness.json").read_text())

..................                                                       [100%]
=============================== warnings summary ===============================
tests/test_gcn.py::test_cora_public_split_and_test_node_order
tests/test_gcn.py::test_cora_public_split_and_test_node_order
tests/test_gcn.py::test_cora_public_split_and_test_node_order
  /private/var/folders/n0/w6gm76jn2qx81f6vm2lhhdl80000gn/T/needle-source-5m8devbb/needle_sparse_gcn/apps/gcn.py:81: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be removed in SciPy 2.0.0.
    return pickle.load(f, encoding="latin1")

tests/test_gcn.py::test_cora_public_split_and_test_node_order
  /private/var/folders/n0/w6gm76jn2qx81f6vm2lhhdl80000gn/T/needle-source-5m8devbb/needle_sparse_gcn/tests/test_gcn.py:111: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be 

Using needle backend
logits                 0
loss                   0
weight1_gradient       0
weight2_gradient       0
weight1_after_step     0
weight2_after_step     0
{
  "seed": 7,
  "dataset": "Cora",
  "dropout": 0,
  "optimizer": "Adam",
  "atol": 1e-05,
  "rtol": 0.0001,
  "dense_loss": 1.9456716775894165,
  "sparse_loss": 1.9456716775894165,
  "max_abs_error": {
    "logits": 0.0,
    "loss": 0.0,
    "weight1_gradient": 0.0,
    "weight2_gradient": 0.0,
    "weight1_after_step": 0.0,
    "weight2_after_step": 0.0
  }
}



## Paired training experiment

Each seed defines two separate worker processes with the same node split, starting weights, dropout random stream and optimizer configuration. One process constructs the CSR adjacency and the other constructs its dense counterpart. The driver alternates which layout runs first across pairs. Both call the same training function and use the native CPU device; the layout determines only how graph aggregation is dispatched. These choices make differences in the loss trajectory interpretable separately from differences in machine timing.

We trained for 100 full-batch epochs at each of three seeds. We measured validation accuracy with dropout disabled after every fifth epoch, keeping the earliest checkpoint with the highest observed validation accuracy. The test split was scored after restoring that checkpoint. Runtime measurements cover one complete epoch and exclude validation, data loading and compilation. Each run reports the median over epochs 6 through 100. Peak resident memory is the operating system's high-water mark for the worker process, including preprocessing, training and evaluation.

The following table reads the six recorded runs directly from the archive. Layouts on the same row pair were run on the same arm64 macOS host. Local reproduction and Colab timings will depend on their respective compiler, operating system and CPU.

In [4]:
from IPython.display import Markdown, display
from statistics import median

runs = json.loads((root / "results/benchmark.json").read_text())["runs"]
header = "| Seed | Layout | Selected epoch | Validation | Test | Median ms/epoch | Peak MiB |"
table = [header, "|---:|:---|---:|---:|---:|---:|---:|"]
for r in sorted(runs, key=lambda x: (x["seed"], x["layout"])):
    table.append(f"| {r['seed']} | {r['layout']} | {r['selected_epoch']} | "
                 f"{r['validation_accuracy']:.1%} | {r['test_accuracy']:.1%} | "
                 f"{1000*r['median_epoch_s']:.2f} | {r['peak_rss_bytes']/2**20:.2f} |")
display(Markdown("\n".join(table)))
for layout in ("dense", "sparse"):
    group = [r for r in runs if r["layout"] == layout]
    print(layout, f"{group[0]['adjacency_bytes']:,} adjacency bytes;",
          f"{1000*median(r['median_epoch_s'] for r in group):.2f} ms/epoch;",
          f"{median(r['peak_rss_bytes'] / 2**20 for r in group):.2f} MiB median peak")

| Seed | Layout | Selected epoch | Validation | Test | Median ms/epoch | Peak MiB |
|---:|:---|---:|---:|---:|---:|---:|
| 7 | dense | 90 | 78.6% | 81.3% | 120.33 | 173.33 |
| 7 | sparse | 90 | 78.6% | 81.3% | 32.15 | 143.80 |
| 19 | dense | 100 | 77.2% | 79.9% | 121.01 | 174.03 |
| 19 | sparse | 100 | 77.2% | 79.9% | 28.65 | 141.70 |
| 41 | dense | 75 | 77.2% | 78.1% | 122.98 | 189.22 |
| 41 | sparse | 75 | 77.2% | 78.1% | 34.00 | 118.55 |

dense 29,333,056 adjacency bytes; 121.01 ms/epoch; 174.03 MiB median peak
sparse 116,948 adjacency bytes; 32.15 ms/epoch; 141.70 MiB median peak


## Interpreting the results

After normalization Cora has 13,264 stored positions among $2,708^2$ possibilities. At about 0.18% density, the index overhead is small compared with the zero entries omitted. Using CSR reduces the adjacency payload by a factor of 250.8. Peak process memory across the three seeds had a median of 141.7 MiB for CSR and 174.0 MiB for dense storage. These quantities answer different questions. A process still holds dense input features, trainable weights, intermediate activations, gradients, optimizer state and its runtime. Changing adjacency layout can remove only part of its working set. The high-water mark can also occur before or after a training step.

The median of the three per-run epoch medians was 32.15 ms for CSR and 121.01 ms for dense storage, a ratio of 3.76. Aggregation benefits directly from the sparsity of the graph. Dense projections, nonlinearities and parameter updates still have the same dimensions and use the same code. The measured ratio describes our end-to-end Needle implementation for these feature widths. It includes any costs of the dense backend's handling of transposes and compact buffers; another dense kernel or a different graph could change it substantially.

Test accuracies for seeds 7, 19 and 41 were respectively 81.3%, 79.9% and 78.1% for both layouts, for an average of 79.8%. Validation selected epochs 90, 100 and 75 respectively. Equality within each pair indicates that this storage change preserved classification behavior in the observed runs. The variation between seeds shows why comparing two independently initialized models could give a misleading impression that a storage layout affects accuracy. Three seeds on one public split are a small sample for evaluating a learning algorithm; they are more informative here as repeated checks of a numerical implementation.

## Reproduction and limits

The short run below exercises data verification, training, validation, checkpoint restoration and testing on the current machine. It uses one seed and five epochs, so its accuracies and time estimates describe a different budget than the full experiment above. Switch on the full reproduction to compare matched 100-epoch runs. Both commands write to a temporary output directory, keeping the recorded measurements available for comparison.

In [5]:
def paired(epochs, warmup, seeds, filename):
    run(python, "-m", "apps.experiment", "--epochs", str(epochs), "--warmup", str(warmup),
        "--seeds", *[str(s) for s in seeds], "--output", str(scratch_dir / filename))

paired(5, 1, [7], "smoke.json")
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    paired(100, 5, [7, 19, 41], "reproduced.json")

Using needle backend
sparse seed 7 | 0.0316 s/epoch | val 0.464 | test 0.460


dense  seed 7 | 0.1267 s/epoch | val 0.464 | test 0.460


The implementation targets two-dimensional float32 operands on the native CPU backend. Compacting a noncontiguous dense input can allocate a temporary buffer. Our tests cover first-order differentiation of stored values and features, including a case where several edges contribute to one node's feature gradient. They do not establish performance on graphs with high degree imbalance, much wider features, or enough nodes to make the dense baseline infeasible. The full-batch GCN also keeps all features resident, which limits how far the savings in adjacency storage can take this training design.

We also isolated forward multiplication on synthetic 512-by-512 matrices with 16 feature columns. The experiment checked each result against dense multiplication and interleaved the two layouts over 25 measured repetitions after warm-up. At 50% density, the CSR and dense medians were about 0.393 and 0.733 ms. At 100% density, they were about 0.796 and 0.733 ms. Dense performance changed little across those two cases, while CSR evaluated twice as many stored entries in the second. The two observations bracket a change in ranking for these settings. Inferring an exact crossover or predicting training speed on a second graph would require finer sampling and measurements of the backward pass. Individual samples and all tested densities are in `results/density_sweep.json`.

## Sources

- [Kipf and Welling, *Semi-Supervised Classification with Graph Convolutional Networks*](https://arxiv.org/abs/1609.02907). GCN architecture and normalized graph propagation.
- [Cora data in the authors' repository, pinned revision](https://github.com/tkipf/gcn/tree/39a4089fe72ad9f055ed6fdb9746abdcfebc4d81/gcn/data). Dataset files and public split. The archive includes a checksum manifest.
- [CMU Deep Learning Systems course project](https://dlsyscourse.org/project/). Project requirements. The framework builds on the team's HW4 Extra Needle implementation; the sparse operators, GCN application, comparisons and recorded measurements belong to this project.